In [1]:
"""
Лабораторная работа: Паттерны «Стратегия», «Наблюдатель» и «Адаптер»
Цель работы
Закрепить навыки проектирования программного обеспечения с использованием 
паттернов Strategy (стратегия), Observer (наблюдатель) и Adapter (адаптер). 
Научиться применять эти паттерны для создания гибких, расширяемых и 
слабосвязанных систем на примере модуля обработки и оповещения в 
медицинской информационной системе.

Постановка задачи
Необходимо разработать программный модуль, который:

получает данные от различных медицинских устройств (старых и новых) через 
адаптеры, унифицирующие интерфейсы;

обрабатывает эти данные (например, фильтрует шумы, вычисляет тренды) с 
помощью взаимозаменяемых стратегий;

оповещает заинтересованные компоненты (врачи, мониторы, архив) при 
наступлении критических событий через паттерн Наблюдатель.

Система должна поддерживать:

подключение устройств с разными протоколами через адаптеры;

переключение алгоритмов обработки сигнала во время выполнения (стратегии);

динамическую подписку/отписку наблюдателей на события (превышение порогов, сбой устройства).

Компоненты системы
1. Адаптер (Adapter)
Target – интерфейс IDeviceDataProvider, определяющий метод 
get_patient_data() (возвращает структурированные данные: пульс, давление, SpO₂).

Adaptee – два класса старых устройств:

OldMonitor – выдаёт строку в формате "HR:75, BP:120/80, O2:98".

LegacyVentilator – выдаёт массив байт [0x48, 0x52, ...].

Adapter – классы OldMonitorAdapter и LegacyVentilatorAdapter, 
которые преобразуют данные старых устройств в унифицированный формат PatientData.

2. Стратегия (Strategy)
Контекст – класс DataProcessor, который принимает стратегию и применяет её к данным.

Стратегия – интерфейс IProcessingStrategy с 
методом process(data: PatientData) -> ProcessedData.

Конкретные стратегии:

NoiseFilterStrategy – удаляет выбросы 
(например, заменяет аномальные значения на предыдущие).

TrendAnalysisStrategy – вычисляет скользящее среднее за последние 5 измерений.

AlertThresholdStrategy – проверяет превышение 
порогов (пульс > 120 или < 50) и генерирует событие (через наблюдателя).

3. Наблюдатель (Observer)
Наблюдаемый субъект – класс HealthMonitor 
(или сам DataProcessor), который управляет 
подпиской и уведомляет наблюдателей при возникновении событий.

Наблюдатель – интерфейс IAlertObserver с методом update(event_type, description).

Конкретные наблюдатели:

DoctorScreenObserver – выводит сообщение на экран врача.

CentralArchiveObserver – записывает событие в лог (имитация сохранения в БД).

EmergencyCallObserver – при критическом событии вызывает метод call_emergency().

Требования к реализации
Адаптер:

Реализовать целевой интерфейс IDeviceDataProvider.

Создать два адаптера для OldMonitor и LegacyVentilator.

Продемонстрировать, что клиентский код работает 
с любым устройством через единый интерфейс.

Стратегия:

Определить интерфейс IProcessingStrategy.

Реализовать три конкретные стратегии (фильтр шумов, анализ тренда, проверка порогов).

В классе DataProcessor предусмотреть метод set_strategy() 
для смены алгоритма во время выполнения.

Наблюдатель:

Реализовать механизм подписки в HealthMonitor 
(методы attach(), detach(), notify()).

Создать минимум двух наблюдателей, один из которых 
реагирует на события от стратегии проверки порогов.

Показать, что при возникновении критических значений наблюдатели получают уведомления.

Клиентский код (run_medical_system()):

Создать экземпляры устройств и обернуть их в адаптеры.

Настроить конвейер обработки: данные → адаптер → стратегия → наблюдатель.

Продемонстрировать смену стратегии в рантайме.

Сымитировать поступление нескольких измерений и вызвать критическое событие.

Примечания
Для имитации поступления данных можно использовать простой 
цикл или генератор случайных значений.

События (например, "CRITICAL_HR") могут передаваться в метод update() наблюдателя.
"""

_IncompleteInputError: incomplete input (969763094.py, line 1)

In [ ]:
"""
Лабораторная работа: Паттерны «Стратегия», «Наблюдатель» и «Адаптер»
Тема: Модуль обработки данных и оповещения в медицинской информационной системе

Заполните пропуски (TODO) в коде, чтобы реализовать:
- Адаптеры для старых устройств (OldMonitor, LegacyVentilator)
- Стратегии обработки данных (фильтр шумов, анализ тренда, проверка порогов)
- Наблюдателей (экран врача, архив, экстренный вызов)
- Клиентский код, демонстрирующий все паттерны
"""

from abc import ABC, abstractmethod
from typing import List, Optional, Tuple
from dataclasses import dataclass
import random

# ==================== ВСПОМОГАТЕЛЬНЫЕ КЛАССЫ ДАННЫХ ====================

@dataclass
class PatientData:
    """Унифицированный формат данных о пациенте."""
    heart_rate: int       # пульс (уд/мин)
    systolic_bp: int      # систолическое давление
    diastolic_bp: int     # диастолическое давление
    oxygen_saturation: int  # SpO₂ (%)

@dataclass
class ProcessedData:
    """Результат обработки данных стратегией."""
    is_critical: bool          # превышен ли критический порог?
    message: str               # сообщение для наблюдателей
    extra_info: dict = None    # дополнительные данные (тренд, скользящее среднее и т.д.)

# ==================== ПАТТЕРН «АДАПТЕР» ====================

class IDeviceDataProvider(ABC):
    """Целевой интерфейс, ожидаемый клиентским кодом."""
    @abstractmethod
    def get_patient_data(self) -> PatientData:
        """Получить структурированные данные о пациенте."""
        pass

# --- Адаптируемые классы (старые устройства) ---

class OldMonitor:
    """Старый монитор, выдаёт данные в виде строки."""
    def read_data(self) -> str:
        # Имитация получения строки от устройства
        return f"HR:{random.randint(60, 130)}, BP:{random.randint(90, 150)}/{random.randint(60, 100)}, O2:{random.randint(90, 100)}"

class LegacyVentilator:
    """Старый аппарат ИВЛ, выдаёт массив байт."""
    def get_byte_stream(self) -> bytes:
        # Имитация байтового потока
        hr = random.randint(60, 130)
        sbp = random.randint(90, 150)
        dbp = random.randint(60, 100)
        spo2 = random.randint(90, 100)
        return f"HR:{hr}, BP:{sbp}/{dbp}, O2:{spo2}".encode()

# --- Адаптеры (нужно заполнить) ---

class OldMonitorAdapter(IDeviceDataProvider):
    """Адаптер для OldMonitor: преобразует строку в PatientData."""
    def __init__(self, monitor: OldMonitor):
        self.monitor = monitor

    def get_patient_data(self) -> PatientData:
        # TODO: Получить строку от self.monitor.read_data(), распарсить её
        # и вернуть объект PatientData.
        pass

class LegacyVentilatorAdapter(IDeviceDataProvider):
    """Адаптер для LegacyVentilator: преобразует байты в PatientData."""
    def __init__(self, ventilator: LegacyVentilator):
        self.ventilator = ventilator

    def get_patient_data(self) -> PatientData:
        # TODO: Получить байты от self.ventilator.get_byte_stream(), декодировать,
        # распарсить и вернуть PatientData.
        pass

# ==================== ПАТТЕРН «НАБЛЮДАТЕЛЬ» ====================

class IAlertObserver(ABC):
    """Интерфейс наблюдателя (получателя уведомлений)."""
    @abstractmethod
    def update(self, event_type: str, description: str):
        """Вызывается наблюдаемым объектом при наступлении события."""
        pass

class HealthMonitor:
    """Наблюдаемый объект (субъект). Управляет подпиской и рассылкой уведомлений."""
    def __init__(self):
        self._observers: List[IAlertObserver] = []

    def attach(self, observer: IAlertObserver):
        """Подписать наблюдателя."""
        # TODO: Добавить observer в список, избегая дублирования
        pass

    def detach(self, observer: IAlertObserver):
        """Отписать наблюдателя."""
        # TODO: Удалить observer из списка, если он там есть
        pass

    def notify(self, event_type: str, description: str):
        """Оповестить всех подписанных наблюдателей."""
        # TODO: Вызвать update(event_type, description) для каждого observer
        pass

# --- Конкретные наблюдатели (нужно заполнить) ---

class DoctorScreenObserver(IAlertObserver):
    """Наблюдатель, выводящий сообщение на экран врача."""
    def update(self, event_type: str, description: str):
        # TODO: Вывести сообщение в консоль в формате:
        # [ЭКРАН ВРАЧА] Событие: <event_type> - <description>
        pass

class CentralArchiveObserver(IAlertObserver):
    """Наблюдатель, записывающий событие в лог (имитация БД)."""
    def update(self, event_type: str, description: str):
        # TODO: Добавить запись в файл "alerts.log" или просто вывести в консоль с пометкой [АРХИВ]
        pass

class EmergencyCallObserver(IAlertObserver):
    """Наблюдатель, вызывающий экстренную службу при критическом событии."""
    def update(self, event_type: str, description: str):
        # TODO: Если event_type == "CRITICAL", то вызвать метод call_emergency()
        pass

    def call_emergency(self):
        print("[ЭКСТРЕННЫЙ ВЫЗОВ] Набрать 103! Отправлена бригада реанимации.")

# ==================== ПАТТЕРН «СТРАТЕГИЯ» ====================

class IProcessingStrategy(ABC):
    """Интерфейс стратегии обработки данных."""
    @abstractmethod
    def process(self, data: PatientData, history: List[PatientData]) -> ProcessedData:
        """
        Обработать текущие данные, возможно, используя историю.
        Возвращает объект ProcessedData.
        """
        pass

# --- Конкретные стратегии (нужно заполнить) ---

class NoiseFilterStrategy(IProcessingStrategy):
    """Фильтр шумов: заменяет выбросы на предыдущие значения."""
    def process(self, data: PatientData, history: List[PatientData]) -> ProcessedData:
        # TODO: Если history не пуста, сравнить каждый параметр с предыдущим.
        # Если разница слишком велика (например, пульс изменился более чем на 30),
        # то заменить на предыдущее значение и вернуть сообщение о коррекции.
        # В остальных случаях вернуть исходные данные с is_critical=False.
        pass

class TrendAnalysisStrategy(IProcessingStrategy):
    """Анализ тренда: вычисляет скользящее среднее за последние 5 измерений."""
    def process(self, data: PatientData, history: List[PatientData]) -> ProcessedData:
        # TODO: Взять последние 5 записей (включая текущую) и вычислить среднее
        # для пульса, давления, SpO2. Вернуть ProcessedData с extra_info = {"trend": ...}
        pass

class AlertThresholdStrategy(IProcessingStrategy):
    """Проверка превышения порогов: генерирует критическое событие."""
    def process(self, data: PatientData, history: List[PatientData]) -> ProcessedData:
        # TODO: Если heart_rate > 120 или heart_rate < 50, или oxygen_saturation < 88,
        # то вернуть ProcessedData(is_critical=True, message="...")
        # Иначе is_critical=False.
        pass

# ==================== КОНТЕКСТ ДЛЯ СТРАТЕГИИ ====================

class DataProcessor:
    """Контекст, использующий стратегию. Также может уведомлять наблюдателей."""
    def __init__(self, strategy: IProcessingStrategy, monitor: HealthMonitor):
        self._strategy = strategy
        self._history: List[PatientData] = []   # сохраняем последние данные
        self.monitor = monitor

    def set_strategy(self, strategy: IProcessingStrategy):
        """Позволяет менять стратегию во время выполнения."""
        self._strategy = strategy

    def process_data(self, data: PatientData) -> ProcessedData:
        """Обработать данные с помощью текущей стратегии и сохранить историю."""
        result = self._strategy.process(data, self._history)
        self._history.append(data)
        # Сохраняем не более 20 записей
        if len(self._history) > 20:
            self._history.pop(0)

        # Если результат содержит критическое событие, уведомить наблюдателей
        if result.is_critical:
            self.monitor.notify("CRITICAL", result.message)
        else:
            self.monitor.notify("INFO", result.message)
        return result

# ==================== КЛИЕНТСКИЙ КОД (ДЕМОНСТРАЦИЯ) ====================

def run_medical_system():
    print("=== МЕДИЦИНСКАЯ СИСТЕМА ОБРАБОТКИ ДАННЫХ И ОПОВЕЩЕНИЯ ===\n")

    # --- 1. Создаём наблюдателей и подписываем их ---
    health_monitor = HealthMonitor()
    # TODO: Создать экземпляры DoctorScreenObserver, CentralArchiveObserver, EmergencyCallObserver
    # TODO: Подписать их через health_monitor.attach()

    # --- 2. Создаём устройства и адаптеры ---
    old_mon = OldMonitor()
    old_vent = LegacyVentilator()
    # TODO: Создать адаптеры OldMonitorAdapter и LegacyVentilatorAdapter

    # --- 3. Создаём стратегии и контекст ---
    # Начнём со стратегии фильтрации шумов
    filter_strategy = NoiseFilterStrategy()
    processor = DataProcessor(filter_strategy, health_monitor)

    # --- 4. Имитация поступления данных от разных устройств ---
    # Используем адаптер для старого монитора
    print("-- Получение данных от OldMonitor через адаптер --")
    for i in range(5):
        data = old_mon.get_patient_data()   # TODO: через адаптер получить PatientData
        # TODO: передать данные в processor.process_data() и вывести результат
        pass

    # --- 5. Смена стратегии во время выполнения ---
    print("\n-- Смена стратегии на анализ тренда --")
    processor.set_strategy(TrendAnalysisStrategy())
    # TODO: снова получить несколько измерений и обработать

    # --- 6. Генерация критического события (например, SpO2 < 88) ---
    print("\n-- Имитация критической гипоксемии --")
    # TODO: вручную создать PatientData с низкой сатурацией и обработать
    # Ожидается, что сработает EmergencyCallObserver

    # --- 7. Вывод истории (опционально) ---
    print("\n--- История обработанных данных ---")
    # TODO: вывести содержимое processor._history (можно добавить метод в DataProcessor)

if __name__ == "__main__":
    run_medical_system()